# Misconception Lab

This executable notebook is the offline verification artifact for the SPEED August AI Challenge prototype. It exercises the same grounded tutor service used by `app.py`: learner answer → misconception signal → Socratic hint → revised answer → mastery.

Run it from the project root. The notebook uses no network, model API, or hidden credentials.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    candidates = [ROOT, *ROOT.parents]
    ROOT = next(path for path in candidates if (path / 'src').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.tutor import TutorService

service = TutorService(mode='demo')
print(f'Lesson source: {service.lesson.source}')
print(f'Lesson chunks: {len(service.lesson_chunks)}')

In [ ]:
question = 'Solve 3x + 5 = 20. What is x?'
scenarios = {
    'diagnostic_misconception': ('x = 15', 1),
    'guided_revision': ('x = 5 because I subtracted 5 from both sides, then divided by 3.', 2),
    'answer_only_request': ('Just give me the answer.', 1),
    'bounded_off_topic': ('What is the weather on Mars?', 1),
}
responses = {}
for name, (answer, hint_level) in scenarios.items():
    responses[name] = service.respond(answer, question=question, hint_level=hint_level)
    response = responses[name]
    print(json.dumps({
        'scenario': name,
        'diagnosis': response['diagnosis'],
        'mastery_score': response['mastery_score'],
        'safety_flags': response['safety_flags'],
    }, indent=2))

In [ ]:
wrong = responses['diagnostic_misconception']
correct = responses['guided_revision']
assert wrong['diagnosis'] == 'misconception'
assert correct['diagnosis'] == 'correct'
assert float(correct['mastery_score']) > float(wrong['mastery_score'])
assert 'x = 5' not in wrong['hint'].lower()
for response in responses.values():
    retrieved_ids = {str(chunk['chunk_id']) for chunk in response['retrieved_chunks']}
    citation_ids = {str(citation['chunk_id']) for citation in response['citations']}
    assert citation_ids <= retrieved_ids
    assert 0.0 <= float(response['mastery_score']) <= 1.0
print('Public contract and grounding assertions passed.')

In [ ]:
from evals.run_eval import main

assert main() == 0
print('Golden evaluation passed: 5/5.')

## Run the product

From the project root, install `requirements.txt` and run `streamlit run app.py`. Demo mode is deterministic and needs no API key. The Devpost submission should include the source tree and a demo video of the interaction; this notebook is the reproducible verification companion.